In [11]:

# ============================================================
# NOTEBOOK 06 — NDCLD15 GAN AUGMENTATION
# Unconditional DCGAN-style experiment
# ============================================================
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time, zipfile, random, os, copy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image, make_grid

START = time.time()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/kaggle/working/nd_iris_project")
IMAGE_DIR = PROJECT / "notebook02_preprocessing/images_224"
OUT = PROJECT / "notebook06_gan_augmentation"
SAMPLE_DIR = OUT / "generated_samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGE_DIR.exists(), f"Missing images: {IMAGE_DIR}"
images = sorted(IMAGE_DIR.glob("*.png"))
assert images, "No preprocessed images found."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
print(f"Device: {device}; available GPUs: {gpu_count}")
for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# 64x64 training resolution keeps GAN training efficient on T4 GPUs.
IMAGE_SIZE = 64
LATENT_DIM = 128
BATCH_SIZE = 128
EPOCHS = 5
LR = 2e-4
WORKERS = min(4, os.cpu_count() or 2)

class GrayDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            im = im.convert("L").resize(
                (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS
            )
            x = np.asarray(im, dtype=np.float32) / 127.5 - 1.0
        return torch.from_numpy(x[None, :, :])

loader = DataLoader(
    GrayDataset(images), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=WORKERS, pin_memory=(device.type == "cuda"),
    drop_last=True
)

# Generator: latent vector -> 64x64 grayscale image
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, base=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, base*4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(base*4), nn.ReLU(True),        # 4x4
            nn.ConvTranspose2d(base*4, base*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*2), nn.ReLU(True),        # 8x8
            nn.ConvTranspose2d(base*2, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base), nn.ReLU(True),          # 16x16
            nn.ConvTranspose2d(base, base//2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base//2), nn.ReLU(True),       # 32x32
            nn.ConvTranspose2d(base//2, 1, 4, 2, 1),
            nn.Tanh()                                     # 64x64
        )
    def forward(self, z):
        return self.net(z)

# Discriminator: grayscale image -> real/fake logit
class Discriminator(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, base, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base, base*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base*2, base*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base*4, base*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base*8, 1, 4, 1, 0)
        )
    def forward(self, x):
        return self.net(x).flatten()

G = Generator().to(device)
D = Discriminator().to(device)

# DataParallel can help if batch size is large enough.
if gpu_count > 1:
    G = nn.DataParallel(G)
    D = nn.DataParallel(D)

# Architecture smoke test before training.
with torch.no_grad():
    z_test = torch.randn(4, LATENT_DIM, 1, 1, device=device)
    fake_test = G(z_test)
    logits_test = D(fake_test)
    assert fake_test.shape == (4, 1, IMAGE_SIZE, IMAGE_SIZE), fake_test.shape
    assert logits_test.shape[0] == 4, logits_test.shape
print("GAN architecture check: PASSED")

criterion = nn.BCEWithLogitsLoss()
optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
fixed_z = torch.randn(32, LATENT_DIM, 1, 1, device=device)
history = []

for epoch in range(1, EPOCHS + 1):
    G.train()
    D.train()
    sum_d, sum_g, n_batches = 0.0, 0.0, 0
    t0 = time.time()

    for real in tqdm(loader, desc=f"GAN epoch {epoch}/{EPOCHS}"):
        real = real.to(device, non_blocking=True)
        b = real.size(0)
        ones = torch.ones(b, device=device)
        zeros = torch.zeros(b, device=device)

        # Train discriminator
        optD.zero_grad(set_to_none=True)
        z = torch.randn(b, LATENT_DIM, 1, 1, device=device)
        fake = G(z)
        loss_real = criterion(D(real), ones)
        loss_fake = criterion(D(fake.detach()), zeros)
        lossD = 0.5 * (loss_real + loss_fake)
        lossD.backward()
        optD.step()

        # Train generator
        optG.zero_grad(set_to_none=True)
        z = torch.randn(b, LATENT_DIM, 1, 1, device=device)
        generated = G(z)
        lossG = criterion(D(generated), ones)
        lossG.backward()
        optG.step()

        sum_d += lossD.item()
        sum_g += lossG.item()
        n_batches += 1

    mean_d = sum_d / max(n_batches, 1)
    mean_g = sum_g / max(n_batches, 1)
    history.append({
        "epoch": epoch, "discriminator_loss": mean_d,
        "generator_loss": mean_g, "seconds": time.time()-t0
    })
    print(f"Epoch {epoch}: D={mean_d:.4f}, G={mean_g:.4f}")

    with torch.no_grad():
        G.eval()
        preview = G(fixed_z).detach().cpu()
        save_image(
            (preview + 1) / 2,
            SAMPLE_DIR / f"epoch_{epoch:02d}_grid.png",
            nrow=8
        )

    # Save recovery checkpoint after every epoch
    g_core = G.module if isinstance(G, nn.DataParallel) else G
    d_core = D.module if isinstance(D, nn.DataParallel) else D
    torch.save({
        "generator_state_dict": g_core.state_dict(),
        "latent_dim": LATENT_DIM, "image_size": IMAGE_SIZE,
        "epoch": epoch, "history": history
    }, OUT / "NDCLD15_GAN_generator_latest.pth")
    torch.save({
        "discriminator_state_dict": d_core.state_dict(),
        "image_size": IMAGE_SIZE, "epoch": epoch
    }, OUT / "NDCLD15_GAN_discriminator_latest.pth")

# Export final generated sample grid and individual samples.
with torch.no_grad():
    G.eval()
    final_samples = G(torch.randn(64, LATENT_DIM, 1, 1, device=device)).cpu()

save_image((final_samples + 1) / 2, SAMPLE_DIR / "final_generated_grid.png", nrow=8)
for i in range(min(32, len(final_samples))):
    save_image((final_samples[i] + 1) / 2, SAMPLE_DIR / f"final_sample_{i:02d}.png")

hist = pd.DataFrame(history)
hist.to_csv(OUT / "NDCLD15_GAN_training_history.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(hist["epoch"], hist["discriminator_loss"], marker="o", label="Discriminator")
plt.plot(hist["epoch"], hist["generator_loss"], marker="o", label="Generator")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("NDCLD15 GAN training losses")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "NDCLD15_GAN_loss_curve.png", dpi=140)
plt.close()

summary = {
    "source_images": len(images),
    "training_resolution": [IMAGE_SIZE, IMAGE_SIZE],
    "latent_dim": LATENT_DIM,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "final_discriminator_loss": float(hist["discriminator_loss"].iloc[-1]),
    "final_generator_loss": float(hist["generator_loss"].iloc[-1]),
    "device": str(device),
    "gpu_count": gpu_count,
    "conditional_generation": False,
    "verified_contact_lens_labels_available": False,
    "notes": [
        "Unconditional GAN experiment; no lens category labels used.",
        "Loss values alone do not establish realism or diversity.",
        "Generated images require visual inspection and quantitative evaluation."
    ]
}
with open(OUT / "NDCLD15_GAN_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

conclusion = f"""# Notebook 06 — GAN Augmentation: Preliminary Conclusion

- Source images: {len(images):,}
- Training resolution: {IMAGE_SIZE} × {IMAGE_SIZE}, grayscale
- Epochs: {EPOCHS}
- Latent dimension: {LATENT_DIM}
- Final discriminator loss: {summary['final_discriminator_loss']:.6f}
- Final generator loss: {summary['final_generator_loss']:.6f}
- Hardware: {device}; GPUs available: {gpu_count}

An unconditional GAN was trained on the available image subset. Verified contact-lens
labels were unavailable, so generation was not conditioned on lens category. GAN
losses alone cannot establish image realism, diversity, or downstream utility.
The generated samples should be reviewed in Notebook 08 before claims are made.
"""
(OUT / "Notebook06_Conclusion.md").write_text(conclusion)

# ZIP all outputs, checkpoints, plots, history and generated images.
ZIP_PATH = PROJECT / "ND_Iris_Notebook06_Results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook06_gan_augmentation/{p.relative_to(OUT)}")

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    archived = z.namelist()

print("\n" + "=" * 65)
print("NOTEBOOK 06 — GAN RESULTS")
print("=" * 65)
print(json.dumps(summary, indent=2))
print(f"Archived files: {len(archived):,}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"ZIP integrity: {'PASSED' if bad is None else 'FAILED: ' + str(bad)}")
print(f"Generator checkpoint: {OUT / 'NDCLD15_GAN_generator_latest.pth'}")
print(f"Conclusion Markdown: {OUT / 'Notebook06_Conclusion.md'}")
print(f"Results ZIP: {ZIP_PATH}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")
assert bad is None


Device: cuda; available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
GAN architecture check: PASSED


GAN epoch 1/5:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 1: D=0.0724, G=4.6150


GAN epoch 2/5:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 2: D=0.1523, G=6.5621


...

Epoch 3: D=0.2845, G=3.8751


GAN epoch 4/5:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 4: D=0.4073, G=3.1741


GAN epoch 5/5:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 5: D=0.3066, G=3.2427

NOTEBOOK 06 — GAN RESULTS
{
  "source_images": 6233,
  "training_resolution": [
    64,
    64
  ],
  "latent_dim": 128,
  "epochs": 5,
  "batch_size": 128,
  "final_discriminator_loss": 0.3065650798380375,
  "final_generator_loss": 3.2427434225877128,
  "device": "cuda",
  "gpu_count": 2,
  "conditional_generation": false,
  "verified_contact_lens_labels_available": false,
  "notes": [
    "Unconditional GAN experiment; no lens category labels used.",
    "Loss values alone do not establish realism or diversity.",
    "Generated images require visual inspection and quantitative evaluation."
  ]
}
Archived files: 44
ZIP size: 8.55 MB
ZIP integrity: PASSED
Generator checkpoint: /kaggle/working/nd_iris_project/notebook06_gan_augmentation/NDCLD15_GAN_generator_latest.pth
Conclusion Markdown: /kaggle/working/nd_iris_project/notebook06_gan_augmentation/Notebook06_Conclusion.md
Results ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook06_Results.zip
Elapsed: 0

## Conclusion — Notebook 06: GAN Augmentation

### Experimental Setup

An unconditional DCGAN-style model was trained on the available NDCLD15 image subset. All 6,233 source images were converted to grayscale and resized to 64 × 64 pixels for training.

- **Source images:** 6,233
- **Training resolution:** 64 × 64 pixels
- **Latent dimension:** 128
- **Training epochs:** 5
- **Batch size:** 128
- **Hardware:** 2 NVIDIA Tesla T4 GPUs
- **Architecture check:** Passed

### Results

| Epoch | Discriminator loss | Generator loss |
|---|---:|---:|
| 1 | 0.0724 | 4.6150 |
| 2 | 0.1523 | 6.5621 |
| 3 | 0.2845 | 3.8751 |
| 4 | 0.4073 | 3.1741 |
| 5 | 0.3066 | 3.2427 |

The final discriminator loss was **0.3066**, and the final generator loss was **3.2427**. These values describe the training process but do not independently demonstrate image realism or diversity.

### Generated Artifacts

The notebook saved the generator and discriminator checkpoints, generated sample images, training history CSV, loss curve, JSON summary, and `ND_Iris_Notebook06_Results.zip`.

**ZIP integrity:** Passed.  
**Archived files:** 44.

### Limitations

Verified contact-lens class labels were unavailable, so the GAN was trained unconditionally. Its generated images are not conditioned on the No Lens, Soft Lens, or Textured Lens categories.

Training losses alone cannot establish that synthetic images are realistic, diverse, privacy-preserving, or useful for downstream classification. These properties require separate evaluation.

### Overall Conclusion

Notebook 06 successfully trained an unconditional GAN for five epochs and saved the required model artifacts and generated samples. The experiment establishes a preliminary image-generation pipeline for NDCLD15. The quality and potential usefulness of its synthetic images will be examined in subsequent notebooks.